In [5]:
import sys
print("Python executable:", sys.executable)

Python executable: /home/sagemaker-user/medquad-envs/rag-env/bin/python


## MQ-005: AWS S3 Dataset Access and Verification

### 1. Objective

The objective of this experiment is to verify that our AWS SageMaker development environment can securely access and load the preprocessed MedQuAD medical question-answering dataset stored in Amazon S3.

This verification is an essential prerequisite for developing our Retrieval-Augmented Generation (RAG) pipeline using MedGemma and FAISS.

### 2. Research Context

Our MedQuad AI project aims to improve the factual grounding of medical question-answering systems by combining:

- **MedQuAD:** A curated medical question-answering dataset.
- **Sentence Transformers:** An embedding model that converts medical text into numerical vectors.
- **FAISS:** A vector similarity search library for retrieving relevant medical information.
- **MedGemma 4B:** A medical-domain language model intended to generate answers using retrieved evidence.

Before constructing the retrieval pipeline, we must confirm that our processed dataset is accessible and maintains its expected structure and completeness.

### 3. Dataset Information

| Property | Description |
|---|---|
| Dataset | MedQuAD (cleaned version) |
| Storage | Amazon S3 |
| S3 bucket | `medquad-ai-cleaned` |
| Object key | `medquad_cleaned.csv` |
| Expected records | 16,359 |
| Expected features | 4 |
| Columns | `question`, `answer`, `source`, `focus_area` |
| Expected missing values | 0 |

### 4. Methodology

We use the AWS SDK for Python (`boto3`) to access the dataset stored in Amazon S3.

The experiment consists of four steps:

1. Establish an S3 client using the credentials available to the SageMaker environment.
2. Retrieve the cleaned MedQuAD CSV file from the designated S3 bucket.
3. Load the dataset into a Pandas DataFrame.
4. Validate the dataset dimensions, column names, and missing-value count.

### 5. Expected Outcome

The experiment will be considered successful if:

- SageMaker can retrieve the dataset without an authentication or authorization error.
- The dataset contains 16,359 records and four expected columns.
- No missing values are detected.

Successful verification confirms that the dataset is ready for subsequent retrieval preprocessing and vector embedding experiments.

### 6. Next Research Step

Following successful verification, we will prepare the appropriate training knowledge corpus, generate medical-text embeddings, and construct a FAISS vector index.

Held-out evaluation examples will be excluded from the indexed knowledge corpus to reduce evaluation leakage.

In [6]:
import boto3
import pandas as pd
from io import BytesIO

# Connect to AWS S3 using the SageMaker execution role
s3 = boto3.client("s3")

# Load the cleaned MedQuAD dataset
response = s3.get_object(
    Bucket="medquad-ai-cleaned",
    Key="medquad_cleaned.csv"
)

medquad_df = pd.read_csv(
    BytesIO(response["Body"].read())
)

print("Dataset shape:", medquad_df.shape)
print("Columns:", medquad_df.columns.tolist())
print("Missing values:", medquad_df.isnull().sum().sum())
print("Dataset loaded successfully!")

Dataset shape: (16359, 4)
Columns: ['question', 'answer', 'source', 'focus_area']
Missing values: 0
Dataset loaded successfully!


## Experimental Results: S3 Dataset Verification

The cleaned MedQuAD dataset was successfully retrieved from
Amazon S3 using the AWS SageMaker development environment.

### Observations

- S3 connectivity: Successful
- Total records: 16,359
- Total columns: 4
- Missing values: 0
- Dataset integrity: Basic structural checks passed

### Conclusion

The experiment confirms that the SageMaker environment can
access the preprocessed MedQuAD dataset stored in Amazon S3.

The dataset has the expected dimensions and contains no
missing values.

The next stage involves preparing a retrieval knowledge corpus
and generating text embeddings for FAISS-based similarity search.

In [7]:
# MQ-005: Inspect MedQuAD data before RAG preprocessing

print("Dataset preview:")
display(medquad_df.head(3))

print("\nUnique sources:")
print(medquad_df["source"].value_counts())

print("\nDuplicate questions:")
print(medquad_df["question"].duplicated().sum())

Dataset preview:


,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in o...,NIHSeniorHealth,Glaucoma



Unique sources:
source
GHR                  5430
GARD                 5389
NIDDK                1144
NINDS                1088
MPlusHealthTopics     981
NIHSeniorHealth       769
CancerGov             729
NHLBI                 559
CDC                   270
Name: count, dtype: int64

Duplicate questions:
1380


In [8]:
# MQ-005: Analyze repeated questions

question_counts = medquad_df["question"].value_counts()

repeated_questions = question_counts[
    question_counts > 1
]

print("Unique questions:", medquad_df["question"].nunique())

print("Questions appearing more than once:",
      len(repeated_questions))

print("Maximum repetitions:",
      repeated_questions.max())

# Check whether repeated questions have different answers
answer_variations = (
    medquad_df.groupby("question")["answer"]
    .nunique()
)

print("Repeated questions with different answers:",
      ((question_counts > 1) & (answer_variations > 1)).sum())

Unique questions: 14979
Questions appearing more than once: 875
Maximum repetitions: 19


Repeated questions with different answers: 875


### Dataset Duplication Analysis

The cleaned MedQuAD dataset contains 16,359 records
from nine medical information sources.

An initial inspection identified 1,380 repeated-question
rows, indicating that some questions occur multiple times.

Further analysis is required to determine whether these
questions correspond to different answers or medical topics.

This investigation is important for preventing retrieval
and evaluation leakage in the RAG pipeline.

## MQ-005: Duplicate-Question Analysis

### Objective
Identify repeated questions in the cleaned MedQuAD dataset and determine
whether they contain different medical answers.

### Research Importance
Repeated questions may refer to different medical conditions or have
different answers. Removing them without analysis could result in
information loss.

This analysis will help us:
- Identify repeated questions.
- Measure how frequently questions are repeated.
- Detect questions associated with multiple answers.
- Prepare a reliable knowledge corpus for FAISS indexing.
- Reduce the risk of data leakage during RAG evaluation.

In [9]:
# MQ-005: Duplicate-Question Analysis

# Count occurrences of each question
question_counts = medquad_df["question"].value_counts()

# Identify questions occurring more than once
repeated_questions = question_counts[question_counts > 1]

# Count distinct answers associated with each question
answer_variations = (
    medquad_df.groupby("question")["answer"]
    .nunique()
)

# Identify repeated questions with different answers
questions_with_different_answers = (
    repeated_questions.index[
        answer_variations.loc[repeated_questions.index] > 1
    ]
)

# Display results
print("Total records:", len(medquad_df))
print("Unique questions:", medquad_df["question"].nunique())
print("Duplicate-question rows:", medquad_df["question"].duplicated().sum())
print("Questions appearing more than once:", len(repeated_questions))
print("Maximum repetitions of one question:", repeated_questions.max())
print(
    "Repeated questions with different answers:",
    len(questions_with_different_answers)
)

# Show sample repeated questions
print("\nTop 10 repeated questions:")
display(repeated_questions.head(10).to_frame(name="frequency"))

# Show sample questions with multiple answers
print("\nExamples of repeated questions with different answers:")
display(
    medquad_df[
        medquad_df["question"].isin(questions_with_different_answers[:3])
    ][["question", "answer", "source", "focus_area"]]
    .assign(answer=lambda df: df["answer"].str.slice(0, 250) + "...")
)

Total records: 16359
Unique questions: 14979
Duplicate-question rows: 1380
Questions appearing more than once: 875
Maximum repetitions of one question: 19
Repeated questions with different answers: 875

Top 10 repeated questions:


,frequency
question,
What is (are) High Blood Cholesterol ?,19
What is (are) Medicare and Continuing Care ?,14
What is (are) Stroke ?,13
What is (are) Skin Cancer ?,13
What is (are) Breast Cancer ?,12
What are the treatments for Breast Cancer ?,12
What is (are) Colorectal Cancer ?,12
What is (are) High Blood Pressure ?,11
What is (are) Parkinson's Disease ?,11



Examples of repeated questions with different answers:


,question,answer,source,focus_area
122,What is (are) Medicare and Continuing Care ?,Medicare is a federal health insurance program...,NIHSeniorHealth,Medicare and Continuing Care
123,What is (are) Medicare and Continuing Care ?,Medicare Part A is hospital insurance that hel...,NIHSeniorHealth,Medicare and Continuing Care
124,What is (are) Medicare and Continuing Care ?,Medicare Part B is medical insurance. It helps...,NIHSeniorHealth,Medicare and Continuing Care
125,What is (are) Medicare and Continuing Care ?,"Medicare Advantage Plans, sometimes known as M...",NIHSeniorHealth,Medicare and Continuing Care
126,What is (are) Medicare and Continuing Care ?,Medicare Part D helps pay for medications that...,NIHSeniorHealth,Medicare and Continuing Care
127,What is (are) Medicare and Continuing Care ?,Original Medicare is managed by the Federal go...,NIHSeniorHealth,Medicare and Continuing Care
128,What is (are) Medicare and Continuing Care ?,People who choose Original Medicare may wish t...,NIHSeniorHealth,Medicare and Continuing Care
129,What is (are) Medicare and Continuing Care ?,Some people think that Medicare and Medicaid a...,NIHSeniorHealth,Medicare and Continuing Care
130,What is (are) Medicare and Continuing Care ?,Home health care is short-term skilled care at...,NIHSeniorHealth,Medicare and Continuing Care
131,What is (are) Medicare and Continuing Care ?,Here are questions to ask when considering a h...,NIHSeniorHealth,Medicare and Continuing Care


## MQ-005: Evaluation Leakage Assessment

### Objective
Identify question-level overlap between the cleaned MedQuAD
dataset and the existing training, validation, and test sets.

### Research Importance
If evaluation questions and their reference answers are
included in the FAISS knowledge index, the evaluation may
overestimate the system's ability to generalize.

We will preserve medically relevant repeated answers while
establishing a leakage-aware retrieval corpus.

In [10]:
import pandas as pd
from io import BytesIO

s3 = boto3.client("s3")
bucket = "medquad-ai-cleaned"

def load_split(key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(BytesIO(obj["Body"].read()))

train_df = load_split("train.csv")
validation_df = load_split("validation.csv")
test_df = load_split("test.csv")

def normalized_questions(df):
    return set(
        df["question"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.casefold()
    )

train_q = normalized_questions(train_df)
val_q = normalized_questions(validation_df)
test_q = normalized_questions(test_df)
all_q = normalized_questions(medquad_df)

print("Train records:", len(train_df))
print("Validation records:", len(validation_df))
print("Test records:", len(test_df))

print("\nQuestion overlap:")
print("Train vs validation:", len(train_q & val_q))
print("Train vs test:", len(train_q & test_q))
print("Validation vs test:", len(val_q & test_q))

print("\nEvaluation questions found in cleaned dataset:")
print("Validation:", len(val_q & all_q), "/", len(val_q))
print("Test:", len(test_q & all_q), "/", len(test_q))

# Candidate knowledge corpus excluding evaluation questions
excluded_questions = val_q | test_q

rag_corpus = medquad_df[
    ~medquad_df["question"]
    .astype(str)
    .str.strip()
    .str.casefold()
    .isin(excluded_questions)
].copy()

print("\nCandidate RAG corpus records:", len(rag_corpus))

Train records: 400
Validation records: 50
Test records: 50

Question overlap:
Train vs validation: 0
Train vs test: 0
Validation vs test: 0

Evaluation questions found in cleaned dataset:
Validation: 50 / 50
Test: 50 / 50

Candidate RAG corpus records: 16214


## Experimental Results: Evaluation Leakage Assessment

### Dataset Splits
- Training: 400 records
- Validation: 50 records
- Testing: 50 records

### Question-Level Overlap
- Training vs. validation: 0
- Training vs. testing: 0
- Validation vs. testing: 0

### Retrieval Corpus Preparation
All 100 evaluation questions were present in the original
cleaned MedQuAD dataset.

After excluding records matching validation and test
questions, the candidate retrieval corpus contains
16,214 records.

### Research Interpretation
The existing dataset splits show no exact normalized
question overlap.

Removing evaluation-question matches from the retrieval
corpus reduces direct evaluation leakage.

However, additional checks for duplicated answers and
semantically similar questions are needed before claiming
the retrieval corpus is leakage-free.

In [11]:
# MQ-005: Exact answer-overlap assessment

def normalize_text(series):
    return (
        series.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.casefold()
    )

evaluation_answers = set(
    normalize_text(
        pd.concat(
            [validation_df["answer"], test_df["answer"]],
            ignore_index=True
        )
    )
)

evaluation_answers.discard("")

corpus_answers = normalize_text(rag_corpus["answer"])

overlap_mask = corpus_answers.isin(evaluation_answers)

print("Candidate corpus records:", len(rag_corpus))
print("Evaluation reference answers:", len(evaluation_answers))
print("Corpus rows matching evaluation answers:", overlap_mask.sum())

# Exclude exact evaluation-answer matches
rag_corpus_safe = rag_corpus.loc[~overlap_mask].copy()

print("Corpus records after answer filtering:", len(rag_corpus_safe))

Candidate corpus records: 16214
Evaluation reference answers: 97
Corpus rows matching evaluation answers: 350
Corpus records after answer filtering: 15864


## Experimental Results: Answer-Level Leakage Assessment

### Objective
Identify exact answer overlap between the evaluation
datasets and the candidate RAG knowledge corpus.

### Results
- Candidate corpus: 16,214 records
- Unique evaluation reference answers: 97
- Records containing overlapping answers: 350
- Remaining corpus: 15,864 records

### Research Findings
Question-level filtering alone did not eliminate all
exact evaluation-answer overlap.

An additional 350 records matched reference answers
from the validation or test datasets.

After removing these records, the retrieval corpus
contains 15,864 question-answer pairs.

### Limitations
Exact text matching cannot identify all paraphrases,
partial answer overlap, or semantically equivalent
medical information.

Therefore, the resulting corpus is considered
exact-match filtered, not completely leakage-free.

In [12]:
# MQ-005: Final RAG corpus verification

print("Final corpus shape:", rag_corpus_safe.shape)

print("Missing values:")
print(rag_corpus_safe.isnull().sum())

print("\nRemaining exact evaluation-question overlap:")

safe_questions = normalized_questions(rag_corpus_safe)

print(
    len(safe_questions & (val_q | test_q))
)

print("\nRemaining exact evaluation-answer overlap:")

remaining_answers = set(
    normalize_text(rag_corpus_safe["answer"])
)

print(
    len(remaining_answers & evaluation_answers)
)

print("\nSource distribution:")
print(rag_corpus_safe["source"].value_counts())

Final corpus shape: (15864, 4)
Missing values:
question      0
answer        0
source        0
focus_area    0
dtype: int64

Remaining exact evaluation-question overlap:
0

Remaining exact evaluation-answer overlap:


0

Source distribution:
source
GARD                 5343
GHR                  5049
NIDDK                1139
NINDS                1080
MPlusHealthTopics     972
NIHSeniorHealth       739
CancerGov             720
NHLBI                 555
CDC                   267
Name: count, dtype: int64


## MQ-005: Verified RAG Knowledge Corpus

### Objective
Preserve the cleaned and exact-match-filtered MedQuAD
knowledge corpus for retrieval experiments.

### Experimental Results
- Original cleaned records: 16,359
- Final retrieval records: 15,864
- Missing values: 0
- Exact evaluation-question overlap: 0
- Exact evaluation-answer overlap: 0
- Medical sources represented: 9

### Research Decision
The filtered corpus will be used as the initial knowledge
base for semantic retrieval using Sentence Transformers
and FAISS.

The original datasets and existing training, validation,
and test splits will remain unchanged.

### Limitations
Exact-match filtering does not guarantee the absence
of semantic or partial-answer leakage. Further evaluation
will be required.

In [13]:
from pathlib import Path

# Save the verified retrieval corpus
output_dir = Path(
    "/home/sagemaker-user/medical_chatbot/data/processed"
)

output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "medquad_rag_corpus.csv"

rag_corpus_safe.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

# Verify saved file
saved_df = pd.read_csv(output_path)

print("Saved to:", output_path)
print("Saved shape:", saved_df.shape)
print("File size (MB):", round(output_path.stat().st_size / 1024**2, 2))

assert saved_df.shape == rag_corpus_safe.shape

print("Corpus saved and verified successfully!")

Saved to: /home/sagemaker-user/medical_chatbot/data/processed/medquad_rag_corpus.csv
Saved shape: (15864, 4)
File size (MB): 20.75
Corpus saved and verified successfully!


## MQ-005: Sentence Transformer Embedding Experiment

### Objective
Verify that a pretrained Sentence Transformer can convert
medical questions and passages into numerical embeddings.

### Baseline Model
sentence-transformers/all-MiniLM-L6-v2

### Methodology
1. Load the pretrained embedding model on CPU.
2. Encode a small sample of medical questions.
3. Inspect the embedding dimensions.
4. Confirm that numerical vectors are generated successfully.

### Research Considerations
The baseline model is not specifically trained for medical
retrieval. Its performance will require evaluation using
held-out medical questions and retrieval metrics.

Long medical answers will be divided into smaller passages
before full-corpus FAISS indexing.

In [14]:
from sentence_transformers import SentenceTransformer

# Load lightweight embedding model on CPU
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

# Select three sample medical questions
sample_questions = rag_corpus_safe["question"].head(3).tolist()

# Generate embeddings
sample_embeddings = embedding_model.encode(
    sample_questions,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding model loaded successfully!")
print("Number of questions:", len(sample_questions))
print("Embedding shape:", sample_embeddings.shape)
print("Embedding dimension:", sample_embeddings.shape[1])
print("Data type:", sample_embeddings.dtype)

/home/sagemaker-user/medquad-envs/rag-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5150.50it/s]


Embedding model loaded successfully!
Number of questions: 3
Embedding shape: (3, 384)
Embedding dimension: 384
Data type: float32


## MQ-005: Medical Passage Chunking

### Objective
Divide long medical answers into smaller passages suitable
for semantic embedding and retrieval.

### Methodology
- Preserve the original medical question.
- Split answer text into overlapping word-based passages.
- Retain source and medical focus-area metadata.
- Assign each passage a unique identifier.

### Research Rationale
Passage chunking reduces the risk of losing information
when long medical answers exceed the embedding model's
input token limit.

Chunk size and overlap are experimental parameters
that may affect retrieval relevance and factual grounding.

In [15]:
# MQ-005: Medical passage chunking prototype

def chunk_answer(text, chunk_size=120, overlap=20):
    words = str(text).split()
    chunks = []

    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_size])

        if chunk:
            chunks.append(chunk)

        if start + chunk_size >= len(words):
            break

    return chunks


sample_df = rag_corpus_safe.head(5)

chunked_records = []

for row_id, row in sample_df.iterrows():
    passages = chunk_answer(row["answer"])

    for chunk_id, passage in enumerate(passages):
        chunked_records.append({
            "record_id": int(row_id),
            "chunk_id": chunk_id,
            "question": row["question"],
            "passage": passage,
            "source": row["source"],
            "focus_area": row["focus_area"]
        })

print("Original records:", len(sample_df))
print("Generated passages:", len(chunked_records))

for record in chunked_records[:2]:
    print("\nChunk ID:", record["chunk_id"])
    print("Question:", record["question"])
    print("Passage:", record["passage"][:300], "...")

Original records: 5
Generated passages: 13

Chunk ID: 0
Question: What is (are) Glaucoma ?
Passage: Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops There are several different types of glaucoma. Most of these involve the drainage system ...

Chunk ID: 1
Question: What is (are) Glaucoma ?
Passage: lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.) In glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. As the fluid builds up, the pressure inside the eye rises. Unless this pressure is controlled, it may cause damage to the ...


## MQ-005: Passage Token-Length Validation

### Objective
Evaluate whether the generated medical passages fit within
the embedding model's maximum sequence length.

### Research Rationale
Word-based chunking does not guarantee that passages fit
within the model's token limit.

Excessively long passages may be truncated, potentially
removing medically relevant information.

This experiment measures passage token lengths before
full-corpus embedding generation.

In [16]:
# MQ-005: Validate passage token lengths

tokenizer = embedding_model.tokenizer

token_lengths = [
    len(
        tokenizer.encode(
            record["passage"],
            add_special_tokens=True,
            truncation=False
        )
    )
    for record in chunked_records
]

max_tokens = embedding_model.max_seq_length

print("Model maximum sequence length:", max_tokens)
print("Number of passages:", len(token_lengths))
print("Minimum tokens:", min(token_lengths))
print("Maximum tokens:", max(token_lengths))
print("Average tokens:", round(sum(token_lengths) / len(token_lengths), 2))

print(
    "Passages exceeding model limit:",
    sum(length > max_tokens for length in token_lengths)
)

Model maximum sequence length: 256
Number of passages: 13
Minimum tokens: 34
Maximum tokens: 185
Average tokens: 143.69
Passages exceeding model limit: 0


## MQ-005: Combined Question–Passage Token Validation

### Objective
Determine whether combining each medical question with its
corresponding answer passage exceeds the embedding model's
maximum input sequence length.

### Research Rationale
Including the medical question provides additional context
for semantic retrieval, particularly when answer passages
contain incomplete references to a disease or condition.

The combined text must fit within the embedding model's
256-token limit to avoid truncation.

### Methodology
1. Combine each question with its answer passage.
2. Tokenize the combined text without truncation.
3. Measure token-length statistics.
4. Identify passages exceeding the model's input limit.

In [17]:
# MQ-005: Combined question and passage validation

combined_texts = [
    f"Question: {record['question']}\n"
    f"Passage: {record['passage']}"
    for record in chunked_records
]

combined_token_lengths = [
    len(
        tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )
    for text in combined_texts
]

print("Model token limit:", max_tokens)
print("Total combined passages:", len(combined_texts))
print("Minimum tokens:", min(combined_token_lengths))
print("Maximum tokens:", max(combined_token_lengths))
print(
    "Average tokens:",
    round(sum(combined_token_lengths) / len(combined_token_lengths), 2)
)
print(
    "Passages exceeding token limit:",
    sum(length > max_tokens for length in combined_token_lengths)
)

Model token limit: 256
Total combined passages: 13
Minimum tokens: 48
Maximum tokens: 199
Average tokens: 157.23
Passages exceeding token limit: 0


## MQ-005: Full-Corpus Medical Passage Preparation

### Objective
Convert the verified MedQuAD retrieval corpus into
smaller medical passages for semantic retrieval.

### Methodology
1. Apply 120-word chunking with 20-word overlap.
2. Preserve question, source, and focus-area metadata.
3. Assign a unique identifier to each passage.
4. Measure combined question-and-passage token lengths.
5. Identify passages exceeding the 256-token model limit.

### Research Considerations
The chunking parameters are experimental and have not
yet been optimized for medical retrieval accuracy.

All passages must be checked for potential truncation
before embedding generation.

In [18]:
# MQ-005: Generate passages for the complete RAG corpus

import pandas as pd

all_chunks = []

for row_id, row in rag_corpus_safe.iterrows():

    passages = chunk_answer(
        row["answer"],
        chunk_size=120,
        overlap=20
    )

    for chunk_id, passage in enumerate(passages):

        all_chunks.append({
            "record_id": int(row_id),
            "chunk_id": chunk_id,
            "question": row["question"],
            "passage": passage,
            "source": row["source"],
            "focus_area": row["focus_area"]
        })

chunks_df = pd.DataFrame(all_chunks)

print("Original records:", len(rag_corpus_safe))
print("Generated passages:", len(chunks_df))
print("Columns:", chunks_df.columns.tolist())
print("Missing values:", chunks_df.isnull().sum().sum())

Original records: 15864
Generated passages: 37865
Columns: ['record_id', 'chunk_id', 'question', 'passage', 'source', 'focus_area']
Missing values: 0


## MQ-005: Full-Corpus Token-Length Validation

### Objective
Evaluate the token lengths of all 37,865 medical passages
after combining each passage with its corresponding question.

### Methodology
- Tokenize the combined question-and-passage text.
- Measure minimum, maximum, and average token lengths.
- Identify passages exceeding the embedding model's
  maximum sequence length of 256 tokens.

### Research Significance
This validation helps prevent silent truncation during
embedding generation and supports reproducible retrieval
experiments.

In [19]:
# MQ-005: Full-corpus token-length validation

combined_texts = [
    f"Question: {q}\nPassage: {p}"
    for q, p in zip(
        chunks_df["question"],
        chunks_df["passage"]
    )
]

# Tokenize in batches to limit memory usage
batch_size = 256
token_lengths = []

for start in range(0, len(combined_texts), batch_size):
    batch = combined_texts[start:start + batch_size]

    encoded = tokenizer(
        batch,
        add_special_tokens=True,
        truncation=False,
        padding=False
    )

    token_lengths.extend(
        len(ids) for ids in encoded["input_ids"]
    )

chunks_df["token_length"] = token_lengths

max_tokens = embedding_model.max_seq_length

print("Total passages:", len(chunks_df))
print("Model token limit:", max_tokens)
print("Minimum tokens:", min(token_lengths))
print("Maximum tokens:", max(token_lengths))
print("Average tokens:", round(sum(token_lengths) / len(token_lengths), 2))

oversized = chunks_df[chunks_df["token_length"] > max_tokens]

print("Passages exceeding limit:", len(oversized))
print(
    "Percentage exceeding limit:",
    round(100 * len(oversized) / len(chunks_df), 2),
    "%"
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (270 > 256). Running this sequence through the model will result in indexing errors


Total passages: 37865
Model token limit: 256
Minimum tokens: 16
Maximum tokens: 371
Average tokens: 151.23
Passages exceeding limit: 388
Percentage exceeding limit: 1.02 %


## MQ-005: Token-Aware Passage Refinement

### Objective
Resolve passages exceeding the embedding model's
maximum sequence length.

### Findings
- Total passages: 37,865
- Maximum observed length: 371 tokens
- Oversized passages: 388 (1.02%)

### Methodology
Apply token-aware splitting to oversized passages,
preserving their question and source metadata.

A conservative 240-token maximum is used to avoid
truncation during embedding generation.

### Research Significance
Token-aware refinement reduces information loss
caused by embedding model input-length restrictions.

In [20]:
# MQ-005: Token-aware refinement of oversized passages

def split_oversized_passage(question, passage, max_length=240, overlap=20):

    prefix = f"Question: {question}\nPassage: "

    prefix_tokens = tokenizer.encode(
        prefix,
        add_special_tokens=False
    )

    passage_tokens = tokenizer.encode(
        passage,
        add_special_tokens=False
    )

    # Reserve space for model special tokens
    available_tokens = max_length - len(prefix_tokens) - 2

    if available_tokens <= overlap:
        raise ValueError("Question is too long for the token budget.")

    step = available_tokens - overlap
    refined = []

    for start in range(0, len(passage_tokens), step):

        token_chunk = passage_tokens[
            start:start + available_tokens
        ]

        text_chunk = tokenizer.decode(
            token_chunk,
            skip_special_tokens=True
        )

        refined.append(text_chunk)

        if start + available_tokens >= len(passage_tokens):
            break

    return refined


refined_records = []

for row in chunks_df.itertuples(index=False):

    if row.token_length <= 240:
        passages = [row.passage]
    else:
        passages = split_oversized_passage(
            row.question,
            row.passage
        )

    for subchunk_id, passage in enumerate(passages):

        refined_records.append({
            "record_id": row.record_id,
            "chunk_id": row.chunk_id,
            "subchunk_id": subchunk_id,
            "question": row.question,
            "passage": passage,
            "source": row.source,
            "focus_area": row.focus_area
        })

refined_df = pd.DataFrame(refined_records)

print("Original passages:", len(chunks_df))
print("Refined passages:", len(refined_df))
print("Additional passages:", len(refined_df) - len(chunks_df))

Original passages: 37865
Refined passages: 38562
Additional passages: 697


In [21]:
refined_texts = [
    f"Question: {q}\nPassage: {p}"
    for q, p in zip(
        refined_df["question"],
        refined_df["passage"]
    )
]

refined_lengths = []

for start in range(0, len(refined_texts), 256):

    encoded = tokenizer(
        refined_texts[start:start + 256],
        add_special_tokens=True,
        truncation=False,
        padding=False,
        verbose=False
    )

    refined_lengths.extend(
        len(ids) for ids in encoded["input_ids"]
    )

refined_df["token_length"] = refined_lengths

print("Final passages:", len(refined_df))
print("Maximum token length:", max(refined_lengths))
print("Passages over 256:", sum(x > 256 for x in refined_lengths))
print("Passages over 240:", sum(x > 240 for x in refined_lengths))
print("Missing values:", refined_df.isnull().sum().sum())

Final passages: 38562
Maximum token length: 240
Passages over 256: 0
Passages over 240: 0
Missing values: 0


## MQ-005: Retrieval Corpus Quality Assessment

### Objective
Identify website-navigation instructions and other
nonmedical artifacts within the refined MedQuAD passages.

### Research Rationale
The MedQuAD dataset includes content extracted from
medical information websites. Some passages may contain
navigation instructions unrelated to medical knowledge.

Such content may reduce retrieval relevance and
introduce unnecessary context into MedGemma responses.

### Methodology
Identify passages containing common website-navigation
phrases and inspect representative examples before
deciding whether cleaning is appropriate.

In [22]:
# MQ-005: Website artifact inspection

artifact_patterns = [
    r"press the escape",
    r"click here",
    r"watch the video",
    r"lower right.hand corner",
    r"print this page",
    r"javascript",
    r"browser window",
    r"navigation menu",
]

pattern = "|".join(artifact_patterns)

artifact_mask = refined_df["passage"].str.contains(
    pattern,
    case=False,
    regex=True,
    na=False
)

artifact_df = refined_df.loc[artifact_mask].copy()

print("Total passages:", len(refined_df))
print("Potential artifact passages:", len(artifact_df))
print(
    "Artifact percentage:",
    round(100 * len(artifact_df) / len(refined_df), 2),
    "%"
)

display(
    artifact_df[
        ["question", "passage", "source"]
    ].head(10)
)

Total passages: 38562
Potential artifact passages: 193
Artifact percentage: 0.5 %


,question,passage,source
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth
1,What is (are) Glaucoma ?,"lower right-hand corner. To reduce the video, ...",NIHSeniorHealth
5,What causes Glaucoma ?,Thats why a comprehensive dilated eye exam is ...,NIHSeniorHealth
6,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in o...,NIHSeniorHealth
7,What are the symptoms of Glaucoma ?,"To enlarge the video, click the brackets in th...",NIHSeniorHealth
9,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth
12,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth
46,How to prevent High Blood Pressure ?,"usual. The more active you are, the more you w...",NIHSeniorHealth
51,How to prevent High Blood Pressure ?,gain. Men should have no more than two drinks ...,NIHSeniorHealth
72,How to prevent High Blood Pressure ?,"fish, poultry, beans, seeds, and nuts, and cho...",NIHSeniorHealth


## MQ-005: Website Artifact Cleaning

### Objective
Remove identifiable website-navigation instructions from
medical passages without deleting medically relevant content.

### Findings
- Total passages: 38,562
- Potential artifact passages: 193
- Flagged percentage: 0.50%

### Research Decision
A conservative text-cleaning approach will be applied.

Passages will not be removed solely because they contain
website-related phrases. Instead, specific nonmedical
instructions will be removed where identifiable.

### Limitations
Automated cleaning may not identify every website artifact.
Manual inspection of representative passages remains necessary.

In [23]:
# MQ-005: Identify artifact patterns and their frequencies

import re

artifact_patterns = {
    "escape_instruction": r"press the escape",
    "click_here": r"click here",
    "video_instruction": r"watch the video",
    "video_corner": r"lower right.hand corner",
    "print_instruction": r"print this page",
    "javascript": r"javascript",
    "browser_window": r"browser window",
    "navigation_menu": r"navigation menu",
}

for name, pattern in artifact_patterns.items():
    count = refined_df["passage"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    ).sum()

    print(f"{name}: {count}")

escape_instruction: 68


click_here: 112
video_instruction: 67
video_corner: 69
print_instruction: 0
javascript: 0
browser_window: 0
navigation_menu: 0


## MQ-005: Website Artifact Context Analysis

### Objective
Examine website-navigation artifacts before applying
text-cleaning transformations.

### Research Rationale
Some passages contain both medically relevant information
and website instructions.

Removing entire passages could cause unnecessary
information loss.

### Methodology
Extract the surrounding text for detected artifacts
to distinguish nonmedical instructions from useful
medical content.

### Cleaning Principle
Preserve medical information and remove only
clearly identifiable navigation instructions.

In [24]:
import re

# Inspect text surrounding website artifacts
inspection_patterns = [
    r"press the escape",
    r"click here",
    r"to enlarge the video",
    r"lower right.hand corner",
]

inspection_regex = re.compile(
    "|".join(inspection_patterns),
    flags=re.IGNORECASE
)

examples = []

for row in artifact_df.itertuples():

    match = inspection_regex.search(row.passage)

    if match:
        start = max(0, match.start() - 100)
        end = min(len(row.passage), match.end() + 150)

        examples.append({
            "question": row.question,
            "source": row.source,
            "matched_phrase": match.group(),
            "context": row.passage[start:end]
        })

inspection_df = pd.DataFrame(examples)

print("Passages with inspected patterns:", len(inspection_df))

display(
    inspection_df[
        ["question", "matched_phrase", "context"]
    ].head(10)
)

Passages with inspected patterns: 186


,question,matched_phrase,context
0,What is (are) Glaucoma ?,To enlarge the video,chamber and bathes and nourishes the nearby ti...
1,What is (are) Glaucoma ?,lower right-hand corner,"lower right-hand corner. To reduce the video, ..."
2,What causes Glaucoma ?,To enlarge the video,ing with your medical doctor. (Watch the anima...
3,What are the symptoms of Glaucoma ?,To enlarge the video,"xam, tonometry, and pachymetry. (Watch the ani..."
4,What are the symptoms of Glaucoma ?,To enlarge the video,"To enlarge the video, click the brackets in th..."
5,What are the treatments for Glaucoma ?,To enlarge the video,. Others help to improve fluid drainage. (Watc...
6,What is (are) Glaucoma ?,To enlarge the video,often protect your eyes against serious vision...
7,How to prevent High Blood Pressure ?,To enlarge the video,"a time, spread throughout the week. (Watch the..."
8,How to prevent High Blood Pressure ?,To enlarge the video,"drink and would like tips on how to cut back,..."
9,How to prevent High Blood Pressure ?,To enlarge the video,e National Institute on Aging. (Watch the vide...


## MQ-005: Conservative Website Artifact Cleaning

### Objective
Remove identifiable website video-player instructions
without deleting medically relevant passages.

### Methodology
1. Preserve the original passage text.
2. Remove narrowly defined video-navigation instructions.
3. Compare original and cleaned passages.
4. Review the results before applying cleaning to the corpus.

### Research Considerations
Automated text cleaning may accidentally remove useful
information. Therefore, transformations must be inspected
before being applied to the complete retrieval corpus.

In [25]:
import re
import pandas as pd

def clean_website_artifacts(text):
    cleaned = str(text)

    # Remove a specific video-player instruction
    cleaned = re.sub(
        r"To enlarge the video, click the brackets in the "
        r"lower right-hand corner\.",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    # Remove a specific instruction for closing the video
    cleaned = re.sub(
        r"To reduce the video, press the Escape "
        r"\(Esc\) button on your keyboard\.",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    # Normalize whitespace
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    return cleaned


# Test cleaning on the first 10 inspected passages
test_samples = artifact_df.head(10).copy()

test_samples["cleaned_passage"] = (
    test_samples["passage"].apply(clean_website_artifacts)
)

test_samples["changed"] = (
    test_samples["passage"] != test_samples["cleaned_passage"]
)

print("Samples inspected:", len(test_samples))
print("Passages changed:", test_samples["changed"].sum())

display(
    test_samples[
        ["question", "passage", "cleaned_passage", "changed"]
    ]
)

Samples inspected: 10
Passages changed: 10


,question,passage,cleaned_passage,changed
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,Glaucoma is a group of diseases that can damag...,True
1,What is (are) Glaucoma ?,"lower right-hand corner. To reduce the video, ...","lower right-hand corner. ) In glaucoma, for st...",True
5,What causes Glaucoma ?,Thats why a comprehensive dilated eye exam is ...,Thats why a comprehensive dilated eye exam is ...,True
6,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in o...,Symptoms of Glaucoma Glaucoma can develop in o...,True
7,What are the symptoms of Glaucoma ?,"To enlarge the video, click the brackets in th...",) A visual acuity test uses an eye chart test ...,True
9,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...","Although open-angle glaucoma cannot be cured, ...",True
12,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,Glaucoma is a group of diseases that can damag...,True
46,How to prevent High Blood Pressure ?,"usual. The more active you are, the more you w...","usual. The more active you are, the more you w...",True
51,How to prevent High Blood Pressure ?,gain. Men should have no more than two drinks ...,gain. Men should have no more than two drinks ...,True
72,How to prevent High Blood Pressure ?,"fish, poultry, beans, seeds, and nuts, and cho...","fish, poultry, beans, seeds, and nuts, and cho...",True


In [31]:
# MQ-005: Validate website artifact cleaning

test_samples = artifact_df.head(10).copy()

test_samples["cleaned_passage"] = (
    test_samples["passage"].apply(clean_website_artifacts)
)

test_samples["changed"] = (
    test_samples["passage"] != test_samples["cleaned_passage"]
)

print("Samples inspected:", len(test_samples))
print("Passages changed:", test_samples["changed"].sum())

for _, row in test_samples.head(5).iterrows():
    print("\nQuestion:", row["question"])
    print("ORIGINAL:", row["passage"][:400])
    print("CLEANED:", row["cleaned_passage"][:400])
    print("-" * 60)

Samples inspected: 10
Passages changed: 10

Question: What is (are) Glaucoma ?
ORIGINAL: Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops There are several different types of glaucoma. Most of these involve the drainage system within the eye. At the front of the eye there is a small space called the anterior chamber. A clear
CLEANED: Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops There are several different types of glaucoma. Most of these involve the drainage system within the eye. At the front of the eye there is a small space called the anterior chamber. A clear
------------------------------------------------------------

Question: What is (are) Glaucoma ?
ORI

In [32]:
# MQ-005: Inspect website instructions before chunking

import re

glaucoma_rows = rag_corpus_safe[
    rag_corpus_safe["question"].eq("What is (are) Glaucoma ?")
]

pattern = re.compile(
    r"To enlarge the video|Watch the animated video",
    flags=re.IGNORECASE
)

for _, row in glaucoma_rows.iterrows():
    answer = str(row["answer"])
    match = pattern.search(answer)

    if match:
        start = max(0, match.start() - 150)
        end = min(len(answer), match.start() + 450)

        print("SOURCE:", row["source"])
        print("ORIGINAL TEXT:")
        print(answer[start:end])
        print("-" * 70)
        break

SOURCE: NIHSeniorHealth
ORIGINAL TEXT:
nterior chamber. A clear fluid flows through this chamber and bathes and nourishes the nearby tissues. (Watch the video to learn more about glaucoma. To enlarge the video, click the brackets in the lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.) In glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. As the fluid builds up, the pressure inside the eye rises. Unless this pressure is controlled, it may cause damage to the optic nerve and other parts of the eye and result in loss of vision. Open-angle Glaucoma The most com
----------------------------------------------------------------------


In [33]:
import re

def clean_medical_answer(text):
    cleaned = str(text)

    # Remove complete parenthesized video-player instructions
    cleaned = re.sub(
        r"\(Watch the (?:animated )?video\b.*?"
        r"To reduce the video, press the Escape "
        r"\(Esc\) button on your keyboard\.\)",
        "",
        cleaned,
        flags=re.IGNORECASE | re.DOTALL
    )

    # Normalize whitespace
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    return cleaned


# Test on the original glaucoma answer
original_answer = answer
cleaned_answer = clean_medical_answer(original_answer)

print("Original length:", len(original_answer))
print("Cleaned length:", len(cleaned_answer))
print("Characters removed:", len(original_answer) - len(cleaned_answer))

print("\nMedical content preserved:")
print("Anterior chamber:", "anterior chamber" in cleaned_answer.lower())
print("Optic nerve:", "optic nerve" in cleaned_answer.lower())

print("\nVideo instruction remaining:")
print("To enlarge the video" in cleaned_answer)

# Show cleaned text around the affected section
position = cleaned_answer.lower().find("a clear fluid")
print("\nCleaned passage:")
print(cleaned_answer[position:position + 450])

Original length: 1846
Cleaned length: 1656
Characters removed: 190

Medical content preserved:
Anterior chamber: True
Optic nerve: True

Video instruction remaining:
False

Cleaned passage:
A clear fluid flows through this chamber and bathes and nourishes the nearby tissues. In glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. As the fluid builds up, the pressure inside the eye rises. Unless this pressure is controlled, it may cause damage to the optic nerve and other parts of the eye and result in loss of vision. Open-angle Glaucoma The most common type of glaucoma is called open-angle glaucoma. In th


## MQ-005: Full-Corpus Website Artifact Cleaning

### Objective
Apply the validated video-instruction cleaning rule to
the complete MedQuAD retrieval corpus.

### Methodology
1. Preserve the original medical answers.
2. Apply the validated cleaning function.
3. Count modified records.
4. Check for empty answers.
5. Retain the original dataset for reproducibility.

### Research Limitation
The cleaning rule targets a specific video-player instruction
pattern. Other website artifacts may remain and require
additional inspection.

In [34]:
# MQ-005: Apply validated cleaning to original answers

rag_corpus_cleaned = rag_corpus_safe.copy()

rag_corpus_cleaned["answer"] = (
    rag_corpus_cleaned["answer"]
    .apply(clean_medical_answer)
)

# Compare original and cleaned answers
changed_mask = (
    rag_corpus_safe["answer"]
    != rag_corpus_cleaned["answer"]
)

print("Original records:", len(rag_corpus_safe))
print("Cleaned records:", len(rag_corpus_cleaned))
print("Records modified:", changed_mask.sum())

print(
    "Empty answers:",
    rag_corpus_cleaned["answer"].str.strip().eq("").sum()
)

print(
    "Missing values:",
    rag_corpus_cleaned.isnull().sum().sum()
)

print(
    "Remaining targeted video instructions:",
    rag_corpus_cleaned["answer"]
    .str.contains(
        "To enlarge the video",
        case=False,
        regex=False
    ).sum()
)

Original records: 15864
Cleaned records: 15864
Records modified: 52
Empty answers: 0
Missing values: 0
Remaining targeted video instructions: 6


## MQ-005: Residual Website Artifact Assessment

### Objective
Identify video-player instructions that were not removed
by the initial cleaning rule.

### Findings
- Total records: 15,864
- Records modified: 52
- Empty answers: 0
- Remaining targeted video instructions: 6

### Methodology
Inspect the context surrounding the remaining instructions
and determine whether additional cleaning rules are needed.

### Research Consideration
Medical information must be preserved during text cleaning.

In [35]:
# MQ-005: Inspect remaining video instructions

remaining_mask = (
    rag_corpus_cleaned["answer"]
    .str.contains(
        "To enlarge the video",
        case=False,
        regex=False,
        na=False
    )
)

remaining_df = rag_corpus_cleaned.loc[remaining_mask]

print("Remaining records:", len(remaining_df))

for idx, row in remaining_df.iterrows():

    text = row["answer"]

    position = text.lower().find("to enlarge the video")

    start = max(0, position - 150)
    end = min(len(text), position + 300)

    print("\nRecord ID:", idx)
    print("Question:", row["question"])
    print("Source:", row["source"])
    print("Context:")
    print(text[start:end])
    print("-" * 70)

Remaining records: 6

Record ID: 17
Question: How to prevent High Blood Pressure ?
Source: NIHSeniorHealth
Context:
n should have no more than one drink a day. If you drink and would like tips on how to cut back, watch the video "How To Cut Back on Your Drinking." (To enlarge the video, click the brackets in the lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.) See how drinking alcohol can affect you as you age. Don't Smoke Smoking injures blood vessel walls and speeds up the process of hardening of the arteries. It
----------------------------------------------------------------------

Record ID: 232
Question: What are the symptoms of Kidney Disease ?
Source: NIHSeniorHealth
Context:
risk for kidney disease, talk to your health care provider about getting tested. (Watch the video to learn more about the symptoms of kidney disease. To enlarge the videos on this page, click the brackets in the lower right-hand corner of the video screen. To r

In [36]:
import re

def clean_medical_answer_v2(text):
    cleaned = str(text)

    # Match complete parenthesized video-player instructions.
    # The instruction must contain both enlargement and Escape controls.
    pattern = (
        r"\("
        r"[^()]{0,600}?"
        r"To enlarge the videos?\b"
        r"[^()]{0,400}?"
        r"To reduce the videos?\b"
        r"[^()]{0,100}?"
        r"Escape\s*\(Esc\)"
        r"[^()]{0,150}?"
        r"\)"
    )

    cleaned = re.sub(
        pattern,
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    return cleaned
    

In [37]:
# Test the updated function without modifying the dataset

remaining_test = remaining_df.copy()

remaining_test["cleaned_answer"] = (
    remaining_test["answer"].apply(clean_medical_answer_v2)
)

print("Remaining records tested:", len(remaining_test))

print(
    "Records modified:",
    (
        remaining_test["answer"]
        != remaining_test["cleaned_answer"]
    ).sum()
)

print(
    "Remaining enlargement instructions:",
    remaining_test["cleaned_answer"]
    .str.contains(
        "To enlarge the video",
        case=False,
        regex=False
    ).sum()
)

print(
    "Empty answers:",
    remaining_test["cleaned_answer"]
    .str.strip().eq("").sum()
)

# Inspect one example
example = remaining_test.iloc[0]

position = example["cleaned_answer"].lower().find(
    "don't smoke"
)

print("\nCleaned medical context:")
print(example["cleaned_answer"][max(0, position - 150):position + 200])

Remaining records tested: 6
Records modified: 6
Remaining enlargement instructions: 0
Empty answers: 0

Cleaned medical context:
drink and would like tips on how to cut back, watch the video "How To Cut Back on Your Drinking." See how drinking alcohol can affect you as you age. Don't Smoke Smoking injures blood vessel walls and speeds up the process of hardening of the arteries. It increases your chances of stroke, heart disease, peripheral arterial disease, and several form


## MQ-005: Final Medical Answer Cleaning

### Objective
Apply the validated website-artifact cleaning rules to
the complete retrieval corpus before passage chunking.

### Methodology
1. Preserve the original retrieval dataset.
2. Apply both validated cleaning rules sequentially.
3. Verify record counts and missing values.
4. Confirm that targeted video instructions are removed.
5. Preserve the cleaned dataset for reproducibility.

### Research Significance
Cleaning before chunking prevents website instructions
from being fragmented across retrieval passages.

In [38]:
# MQ-005: Apply validated cleaning rules

rag_corpus_final = rag_corpus_safe.copy()

rag_corpus_final["answer"] = (
    rag_corpus_final["answer"]
    .apply(clean_medical_answer)
    .apply(clean_medical_answer_v2)
)

changed_mask = (
    rag_corpus_safe["answer"] != rag_corpus_final["answer"]
)

remaining_video = (
    rag_corpus_final["answer"]
    .str.contains(
        r"To enlarge the videos?",
        case=False,
        regex=True,
        na=False
    )
    .sum()
)

print("Original records:", len(rag_corpus_safe))
print("Final records:", len(rag_corpus_final))
print("Records modified:", changed_mask.sum())
print("Empty answers:", rag_corpus_final["answer"].str.strip().eq("").sum())
print("Missing values:", rag_corpus_final.isnull().sum().sum())
print("Remaining targeted video instructions:", remaining_video)

assert len(rag_corpus_final) == len(rag_corpus_safe)
assert rag_corpus_final["answer"].str.strip().ne("").all()
assert rag_corpus_final.isnull().sum().sum() == 0

Original records: 15864
Final records: 15864
Records modified: 57
Empty answers: 0
Missing values: 0
Remaining targeted video instructions: 0


## MQ-005: Final Medical Passage Generation

### Objective
Generate retrieval passages from the cleaned MedQuAD
knowledge corpus.

### Dataset
- Cleaned medical records: 15,864
- Modified records: 57
- Missing values: 0
- Remaining targeted video instructions: 0

### Methodology
1. Apply 120-word chunking with 20-word overlap.
2. Preserve medical questions and source metadata.
3. Assign identifiers to each passage.
4. Apply token-aware refinement.
5. Validate all passages against the embedding model's
   256-token input limit.

### Research Significance
Cleaning before chunking reduces the fragmentation of
website-navigation instructions and improves the quality
of the retrieval knowledge base.

In [39]:
# MQ-005: Generate passages from cleaned medical answers

final_chunks = []

for row_id, row in rag_corpus_final.iterrows():

    passages = chunk_answer(
        row["answer"],
        chunk_size=120,
        overlap=20
    )

    for chunk_id, passage in enumerate(passages):
        final_chunks.append({
            "record_id": int(row_id),
            "chunk_id": chunk_id,
            "question": row["question"],
            "passage": passage,
            "source": row["source"],
            "focus_area": row["focus_area"]
        })

final_chunks_df = pd.DataFrame(final_chunks)

print("Cleaned medical records:", len(rag_corpus_final))
print("Generated passages:", len(final_chunks_df))
print("Missing values:", final_chunks_df.isnull().sum().sum())

print(
    "Remaining video instructions:",
    final_chunks_df["passage"]
    .str.contains(
        r"To enlarge the videos?",
        case=False,
        regex=True,
        na=False
    ).sum()
)

Cleaned medical records: 15864
Generated passages: 37847
Missing values: 0
Remaining video instructions: 0


## MQ-005: Final Token-Aware Passage Refinement

### Objective
Ensure all cleaned medical passages fit within the
embedding model's maximum input length of 256 tokens.

### Methodology
1. Combine each medical question with its passage.
2. Measure the token length without truncation.
3. Split oversized passages using token-aware chunking.
4. Preserve medical source and focus-area metadata.
5. Verify that no final passage exceeds 240 tokens.

### Research Significance
This process minimizes silent truncation during embedding
generation while retaining the cleaned medical content.

### Limitations
Token-based splitting may divide sentences. Chunking
parameters will require further retrieval evaluation.

In [40]:
# MQ-005: Final token-aware refinement

final_records = []

for row in final_chunks_df.itertuples(index=False):

    combined = (
        f"Question: {row.question}\n"
        f"Passage: {row.passage}"
    )

    token_count = len(
        tokenizer.encode(
            combined,
            add_special_tokens=True,
            truncation=False,
            verbose=False
        )
    )

    if token_count <= 240:
        passages = [row.passage]
    else:
        passages = split_oversized_passage(
            row.question,
            row.passage,
            max_length=240,
            overlap=20
        )

    for subchunk_id, passage in enumerate(passages):
        final_records.append({
            "record_id": row.record_id,
            "chunk_id": row.chunk_id,
            "subchunk_id": subchunk_id,
            "question": row.question,
            "passage": passage,
            "source": row.source,
            "focus_area": row.focus_area
        })

final_refined_df = pd.DataFrame(final_records)

print("Original cleaned passages:", len(final_chunks_df))
print("Refined passages:", len(final_refined_df))
print(
    "Additional passages:",
    len(final_refined_df) - len(final_chunks_df)
)
print("Missing values:", final_refined_df.isnull().sum().sum())

Original cleaned passages: 37847
Refined passages: 38544
Additional passages: 697
Missing values: 0


In [41]:
# MQ-005: Verify all final passages

final_token_lengths = []

for row in final_refined_df.itertuples(index=False):
    combined = (
        f"Question: {row.question}\n"
        f"Passage: {row.passage}"
    )

    final_token_lengths.append(
        len(
            tokenizer.encode(
                combined,
                add_special_tokens=True,
                truncation=False,
                verbose=False
            )
        )
    )

final_refined_df["token_length"] = final_token_lengths

print("Final passages:", len(final_refined_df))
print("Maximum tokens:", max(final_token_lengths))
print("Passages over 240:", sum(x > 240 for x in final_token_lengths))
print("Passages over 256:", sum(x > 256 for x in final_token_lengths))
print("Missing values:", final_refined_df.isnull().sum().sum())

assert max(final_token_lengths) <= 240
assert final_refined_df.isnull().sum().sum() == 0

print("Final token validation passed!")

Final passages: 38544
Maximum tokens: 240
Passages over 240: 0
Passages over 256: 0
Missing values: 0
Final token validation passed!


## MQ-005: Final Retrieval Corpus Preservation

### Objective
Preserve the validated medical passage dataset for
reproducible semantic retrieval experiments.

### Experimental Results
- Cleaned medical records: 15,864
- Final retrieval passages: 38,544
- Maximum token length: 240
- Passages exceeding model limit: 0
- Missing values: 0

### Research Decision
The finalized passages will serve as the knowledge corpus
for the initial FAISS retrieval experiment.

The baseline embedding model is
sentence-transformers/all-MiniLM-L6-v2.

### Limitations
The corpus has passed exact-match leakage filtering,
but semantic overlap and partial-answer leakage have
not been completely ruled out.

The source documents may also contain outdated medical
information, requiring further factuality evaluation.

In [42]:
from pathlib import Path
import pandas as pd

# Project directory
output_dir = Path(
    "/home/sagemaker-user/medical_chatbot/data/processed"
)

output_dir.mkdir(parents=True, exist_ok=True)

# Save final passage dataset
output_path = output_dir / "medquad_final_passages.csv"

final_refined_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

# Reload for verification
verified_df = pd.read_csv(output_path)

assert len(verified_df) == 38544
assert verified_df.isnull().sum().sum() == 0
assert verified_df["token_length"].max() <= 240

print("File saved:", output_path)
print("Saved passages:", len(verified_df))
print(
    "File size (MB):",
    round(output_path.stat().st_size / 1024**2, 2)
)
print("Final retrieval corpus saved successfully!")

File saved: /home/sagemaker-user/medical_chatbot/data/processed/medquad_final_passages.csv
Saved passages: 38544
File size (MB): 25.82
Final retrieval corpus saved successfully!


## MQ-006: Medical Passage Embedding Generation

### Objective
Convert the 38,544 validated medical passages into
dense numerical vectors for semantic retrieval.

### Embedding Model
sentence-transformers/all-MiniLM-L6-v2

### Configuration
- Embedding dimensions: 384
- Execution device: CPU
- Batch size: 32
- Vector normalization: Enabled
- Data type: float32

### Methodology
Each embedding represents the medical question
and its associated passage.

The normalized embeddings will subsequently be
indexed using FAISS with inner-product similarity,
which is equivalent to cosine similarity for
unit-normalized vectors.

### Research Considerations
Embedding quality and medical relevance will be
evaluated using held-out questions before integrating
the retrieval system with MedGemma.

In [43]:
import numpy as np
from pathlib import Path

# Load the verified passage dataset
passages_df = pd.read_csv(
    "/home/sagemaker-user/medical_chatbot/data/processed/medquad_final_passages.csv"
)

# Construct the exact text used during token validation
embedding_texts = [
    f"Question: {row.question}\nPassage: {row.passage}"
    for row in passages_df.itertuples(index=False)
]

print("Passages to embed:", len(embedding_texts))
print("Embedding model:", embedding_model.get_sentence_embedding_dimension())

# Generate normalized embeddings in CPU batches
passage_embeddings = embedding_model.encode(
    embedding_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    device="cpu"
)

passage_embeddings = np.asarray(
    passage_embeddings,
    dtype=np.float32
)

# Validate output
assert passage_embeddings.shape == (38544, 384)
assert np.isfinite(passage_embeddings).all()

print("\nEmbedding shape:", passage_embeddings.shape)
print("Embedding dtype:", passage_embeddings.dtype)
print(
    "Memory usage (MB):",
    round(passage_embeddings.nbytes / 1024**2, 2)
)

# Save embeddings
embedding_path = Path(
    "/home/sagemaker-user/medical_chatbot/data/processed/"
    "medquad_passage_embeddings.npy"
)

np.save(embedding_path, passage_embeddings)

print("Saved embeddings:", embedding_path)

/tmp/ipykernel_4561/212284302.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding model:", embedding_model.get_sentence_embedding_dimension())


Passages to embed: 38544
Embedding model: 384


Batches: 100%|██████████| 1205/1205 [40:11<00:00,  2.00s/it] 



Embedding shape: (38544, 384)
Embedding dtype: float32
Memory usage (MB): 56.46
Saved embeddings: /home/sagemaker-user/medical_chatbot/data/processed/medquad_passage_embeddings.npy
